In [ ]:
# ==========================================
# STEP 1: SETUP, LOADING, AND PREPROCESSING
# ==========================================

# 1. Install necessary libraries
!pip install transformers[torch] datasets emoji contractions scikit-learn

import pandas as pd
import torch
import preprocessing  # This is uploaded preprocessing.py

# 2. Function to load  text and label files
def load_data(text_file, label_file):
    with open(text_file, 'r', encoding='utf-8') as f:
        texts = f.read().splitlines()
    with open(label_file, 'r', encoding='utf-8') as f:
        labels = [int(line.strip()) for line in f]
    return texts, labels

# Load Train, Val, and Test splits
train_texts, train_labels = load_data('train_text.txt', 'train_labels.txt')
val_texts, val_labels = load_data('val_text.txt', 'val_labels.txt')
test_texts, test_labels = load_data('test_text.txt', 'test_labels.txt')

# 3. Apply your custom preprocessing logic
# This step is crucial for reaching >95% accuracy as it cleans noise.
print("Preprocessing data... please wait.")
train_texts_cleaned = [preprocessing.preprocess_tweet(t) for t in train_texts]
val_texts_cleaned = [preprocessing.preprocess_tweet(t) for t in val_texts]
test_texts_cleaned = [preprocessing.preprocess_tweet(t) for t in test_texts]

# 4. Check a sample to make sure it looks correct
print("\n--- SAMPLE CHECK ---")
print(f"Original: {train_texts[0]}")
print(f"Cleaned:  {train_texts_cleaned[0]}")
print(f"Label:    {train_labels[0]}")
print(f"Total Train Samples: {len(train_texts_cleaned)}")

In [ ]:
# ==========================================
# STEP 2: TOKENIZATION & DATASET CREATION
# ==========================================

from transformers import AutoTokenizer
import torch

# 1. Initialize the Tokenizer from the pre-trained Twitter-RoBERTa model
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define the PyTorch Dataset class
# This bridges your cleaned lists and the model
class TwitterDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,    # Cuts off tweets longer than max_len
            padding=True,       # Adds zeros so all inputs are the same size
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = labels

    def __getitem__(self, idx):
        # Extract the input IDs and Attention Mask for this specific index
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# 3. Create the Dataset objects for each split
# We use the cleaned texts from Step 1
print("Tokenizing datasets... this may take a moment.")
train_dataset = TwitterDataset(train_texts_cleaned, train_labels, tokenizer)
val_dataset = TwitterDataset(val_texts_cleaned, val_labels, tokenizer)
test_dataset = TwitterDataset(test_texts_cleaned, test_labels, tokenizer)

# 4. Verify a single item
sample_item = train_dataset[0]
print("\n--- TOKENIZATION VERIFICATION ---")
print(f"Input IDs shape: {sample_item['input_ids'].shape}")
print(f"Attention Mask shape: {sample_item['attention_mask'].shape}")
print(f"Label Tensor: {sample_item['labels']}")
print("Tokenization complete!")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics
    acc = accuracy_score(labels, predictions)
    # This 'macro' average is the official TweetEval standard your friend mentioned
    f1 = f1_score(labels, predictions, average='macro')
    macro_recall = recall_score(labels, predictions, average='macro')

    return {
        'accuracy': acc,
        'f1': f1,
        'macro_recall': macro_recall
    }

In [ ]:
# ==========================================
# STEP 3: MODEL & TRAINING CONFIGURATION (UPDATED)
# ==========================================

from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, EarlyStoppingCallback

# 1. Load the Twitter-Specific Model (The native speaker)
print("Loading the Twitter-RoBERTa model...")
model = AutoModelForSequenceClassification.from_pretrained(
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    num_labels=3
)

# 2. Optimized Training Arguments for 80%+ Accuracy
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,               # 3 epochs prevents the overfitting seen in your 5-epoch run
    per_device_train_batch_size=32,   # Increased from 16 for better stability
    per_device_eval_batch_size=32,
    learning_rate=2e-5,               # Lower learning rate for precise fine-tuning
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type="cosine",       # Gradually slows down for better precision
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",            # Using your fixed version
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    label_smoothing_factor=0.1,       # Helps the model with subjective Twitter labels
    fp16=True,                        # Required for speed and efficiency on Colab GPU
    report_to="none"
)

# 3. Re-initialize the Trainer with the new model and optimized args
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Slightly tighter patience
)

print("Step 3 Updated! The model and training logic are now optimized for accuracy.")

In [ ]:
# ==========================================
# STEP 4: START TRAINING
# ==========================================

print("Starting training... This will take a few minutes depending on GPU speed.")

# This command triggers the 5-step pipeline execution
trainer.train()

print("\nTraining complete! The best model version has been loaded back into memory.")

In [ ]:
# ==========================================
# STEP 5: DYNAMIC FINAL EVALUATION
# ==========================================
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, f1_score

print("Running dynamic evaluation on test set...")

# 1. Generate predictions from the current model state
# This ensures results match the ACTUAL model currently in memory
output = trainer.predict(test_dataset)
y_pred = np.argmax(output.predictions, axis=-1)
y_true = test_labels

# 2. Extract metrics dynamically from the output
# This pulls the numbers directly from your compute_metrics function
test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average='macro')
test_macro_recall = recall_score(y_true, y_pred, average='macro')

# 3. Professional Output for Report
print("\n" + "="*40)
print("       OFFICIAL TEST RESULTS (DYNAMIC)")
print("="*40)
print(f"Test Accuracy        : {test_acc:.4f}")
print(f"Test F1-Score (Macro): {test_f1:.4f}")
print(f"Test Recall (Macro)  : {test_macro_recall:.4f} <--- Official Metric")
print("="*40)

target_names = ['Negative', 'Neutral', 'Positive']
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))

# Store the Confusion Matrix for the next Step (Visualization)
cm_data = confusion_matrix(y_true, y_pred)

In [ ]:
# ==========================================
# STEP 6: CONTINUOUS DYNAMIC VISUALIZATIONS
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Data Extraction
history = trainer.state.log_history
train_loss_raw = [x['loss'] for x in history if 'loss' in x]
val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

# Smoothing
train_loss_smoothed = pd.Series(train_loss_raw).rolling(window=3, min_periods=1).mean()
total_steps = len(train_loss_raw)

# 2. Logic to "Connect" the start
# We take the first training loss as the "Start Point" at Step 0
start_loss = train_loss_raw[0]
val_x_axis = [0] + [(total_steps / len(val_loss)) * i for i in range(1, len(val_loss) + 1)]
val_y_axis = [start_loss] + val_loss

# 3. Plotting
plt.figure(figsize=(8, 6))

# Training Line
plt.plot(train_loss_smoothed, label='Training Loss (Trend)', color='#1f77b4', linewidth=2)

# Validation Line (Now starting from Step 0)
plt.plot(val_x_axis, val_y_axis, label='Validation Loss (Continuous)',
         color='#ff7f0e', marker='o', markersize=10, linewidth=3, markevery=[1, 2, 3])

plt.xlim(0, total_steps)
plt.title('Model Convergence: Training vs. Validation', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Training Steps', fontsize=14, fontweight='semibold')
plt.ylabel('Loss Value', fontsize=14, fontweight='semibold')

plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('continuous_loss_chart.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd

# ==========================================
# ERROR ANALYSIS EXPORT
# ==========================================

# 1. Map the numeric labels back to human-readable text
# 0 -> Negative, 1 -> Neutral, 2 -> Positive
label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

# 2. Create the spreadsheet
# We use the 'test_texts_cleaned' list from your earlier steps
error_analysis_df = pd.DataFrame({
    'Tweet_Text': test_texts_cleaned,
    'True_Label': [label_map[label] for label in y_true],
    'Predicted_Label': [label_map[pred] for pred in y_pred]
})

# 3. Add a helper column for him to filter errors quickly
error_analysis_df['Is_Correct'] = error_analysis_df['True_Label'] == error_analysis_df['Predicted_Label']

# 4. Save to CSV file
file_name = 'Sentiment Analysis.csv'
error_analysis_df.to_csv(file_name, index=False)

print(f"Success! The file '{file_name}' has been created.")
print("To download it: Click the folder icon on the left sidebar, find the file, right-click and 'Download'.")